# Setup

In [ ]:
# ComplexHeatmap pkg installation no longer working...
pkgs <- c('circlize','data.table','ggplot2','gtsummary','gt')#'ComplexHeatmap'
#if(!require(ComplexHeatmap)) { devtools::install_github("jokergoo/ComplexHeatmap") } # Special install
lapply(pkgs, \(pkg) { if(!require(pkg, character.only=T)) {install.packages(pkg);require(pkg)}}) |> invisible()

options(datatable.na.strings=c('NA',''))

if(!file.exists('data/derived/analysis_df.csv')) system('gcloud storage cp gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/derived/analysis_df.csv')
data <- fread('data/derived/analysis_df-mesa.csv')[,exam:=as.factor(exam)]

# Summary table of relevant variables

In [ ]:
data[
  ][, .SD, .SDcols=patterns('^age$|bmi$|sex|hdl$|pa$|ahei|ses|drinks|race$|^chr') #.(age, bmi, sex, hdl, pa, mod_pa, mvpa, vig_pa, ahei_score, dash_score, ses_score, drinks_per_week, race, chr17_36804493_G_T)
  ][, has_metabolomics := !is.na(data$TOM_Id)
  #][ is.na(sex), sex := 'Unknown'
  ] |>
  tbl_summary(
    by=sex,
    missing='no',
    statistic = list(
      all_continuous()  ~ c("{mean} ± {sd}"),
      all_categorical() ~ c("{n} ({p}%)"   )
  )) |>
    add_overall() |> add_n() |> as_gt() |> gt:::as.tags.gt_tbl()

# Heatmap of mPCs vs. covars
Note that not all samples contributed to the mPCs — only samples with measurements for all metabolomics methods CP/CN/HP/AN could be used in PCA.

In [ ]:
heatmap_covars <- c(
  paste0('gPC', 1:11),
  'site', 'season', 'month',
  'age', 'bmi', 'sex', 'race',
  'ahei_score', 'dash_score', 'ses_score', 
  'income', 'drinks_per_week', 'smoking')

runs <- CJ(heatmap_covars, mPC=paste0('mPC',1:20), sorted=F)[
  ][, lms    := mapply(mPC,heatmap_covars, FUN = \(mPC,covar) lm(data[[mPC]] ~ data[[covar]]))
  ][, anovas := lapply(lms, anova)
  ][, p      := sapply(anovas, \(res) res$`Pr(>F)`[1])
]
p_mtx <- as.matrix(dcast(runs, heatmap_covars~mPC, value.var='p'), rownames=1)
p_mtx <- p_mtx[,paste0('mPC',1:20)]

# ComplexHeatmap install not working anymore in Terra, so commenting this out.
#Heatmap(-log10(p_mtx),
#  cell_fun = \(j,i,x,y,w,h,col) if(p_mtx[i,j]<0.05) grid.text('*',x,y,gp=gpar(col='white')),
#  col = colorRamp2(c(0,10), c('black','steelblue')),
#  cluster_columns=F, cluster_rows=F,
#  heatmap_legend_param = list(title = 'log10(p)\nfrom lm')
#)
heatmap(-log10(p_mtx), Rowv=NA,Colv=NA)

Define covariate sets to test.

In [ ]:
covar_sets <- list(
  site                    =c('site'                                                                                                                           ),
  `+sex`                  =c('site', 'sex'                                                                                                                    ),
  `+age`                  =c('site', 'sex', 'age'                                                                                                             ),
  `+ses+income`           =c('site', 'sex', 'age', 'ses_score', 'income'                                                                                      ),
  `+drink+smoke+ahei+dash`=c('site', 'sex', 'age', 'ses_score', 'income', 'drinks_per_week', 'smoking', 'ahei_score', 'dash_score'                            ),
  `+race`                 =c('site', 'sex', 'age', 'ses_score', 'income', 'drinks_per_week', 'smoking', 'ahei_score', 'dash_score', 'race'                    ),
  `+gPCs`                 =c('site', 'sex', 'age', 'ses_score', 'income', 'drinks_per_week', 'smoking', 'ahei_score', 'dash_score', 'race', paste0('gPC', 1:5)),
  `-race`                 =c('site', 'sex', 'age', 'ses_score', 'income', 'drinks_per_week', 'smoking', 'ahei_score', 'dash_score',         paste0('gPC', 1:5)))

# Inspect Variable Transformations
HDL vs various transformations of mvpa.

In [ ]:
tmp <- data[exam==1]
tmp <- tmp[
  ][, mvpa_resid             := resid(lm(mvpa       ~ ., tmp[, .SD, .SDcols=        covar_sets$`+gPCs`       ], na.action=na.exclude))
  ][, mvpa_log_resid         := resid(lm(mvpa_log   ~ ., tmp[, .SD, .SDcols=        covar_sets$`+gPCs`       ], na.action=na.exclude))
  ][, mvpa_wins_resid        := resid(lm(mvpa_wins  ~ ., tmp[, .SD, .SDcols=        covar_sets$`+gPCs`       ], na.action=na.exclude))
  ][, mvpa_trunc_resid       := resid(lm(mvpa_trunc ~ ., tmp[, .SD, .SDcols=        covar_sets$`+gPCs`       ], na.action=na.exclude))
  ][, mvpa_resid_nosex       := resid(lm(mvpa       ~ ., tmp[, .SD, .SDcols=setdiff(covar_sets$`+gPCs`,'sex')], na.action=na.exclude))
  ][, mvpa_log_resid_nosex   := resid(lm(mvpa_log   ~ ., tmp[, .SD, .SDcols=setdiff(covar_sets$`+gPCs`,'sex')], na.action=na.exclude))
  ][, mvpa_wins_resid_nosex  := resid(lm(mvpa_wins  ~ ., tmp[, .SD, .SDcols=setdiff(covar_sets$`+gPCs`,'sex')], na.action=na.exclude))
  ][, mvpa_trunc_resid_nosex := resid(lm(mvpa_trunc ~ ., tmp[, .SD, .SDcols=setdiff(covar_sets$`+gPCs`,'sex')], na.action=na.exclude))
]

rbind(tmp[, .(sex, hdl_log, d1=mvpa,            d2=mvpa,                 f='identity'   ) ],
      tmp[, .(sex, hdl_log,    mvpa_resid,         mvpa_resid_nosex,       'resid'      ) ],
      tmp[, .(sex, hdl_log,    mvpa_log,           mvpa_log,               'log'        ) ],
      tmp[, .(sex, hdl_log,    mvpa_log_resid,     mvpa_log_resid_nosex,   'resid_log'  ) ],
      tmp[, .(sex, hdl_log,    mvpa_wins,          mvpa_wins,              'winsor'     ) ],
      tmp[, .(sex, hdl_log,    mvpa_wins_resid,    mvpa_wins_resid_nosex,  'resid_wins' ) ],
      tmp[, .(sex, hdl_log,    mvpa_trunc,         mvpa_trunc,             'trunc'      ) ],
      tmp[, .(sex, hdl_log,    mvpa_trunc_resid,   mvpa_trunc_resid_nosex, 'resid_trunc') ],
      use.names=F)[
    ][, f := factor(f,levels=unique(f)) ] |>
  ggplot(aes(y=hdl_log)) +
  geom_smooth(aes(x=d1           ), method = 'gam', formula = y ~ s(x, bs='cs'), se=T, na.rm=T) +
  geom_smooth(aes(x=d2, color=sex), method = 'gam', formula = y ~ s(x, bs='cs'), se=T, na.rm=T) +
  facet_wrap(vars(f), ncol=2, scales='free') +
  ggtitle(label='Residuals from `mvpa_variable ~ covars_except_sex`')

# Run models
Strategy: run all lm models upfront and examine the results afterward.\
One set of models we want to test is `mPC ~ pa_or_snp_variable + covar_set` for each possible combination of mPC1-10, untransformed PA or SNP variable, and covariate set. The point of this set of models is to see which covariates make the largest difference in the relationship between our variables of interest and the metabolomics, to help figure out which covariates to adjust for.\
Another set of models is `hdl_log ~ pa_or_snp_variable_transformed + covar_set`, for each possible combination of PA or SNP variable, and covariate set. The point of this is to see the main effects, and also see which covariates make the largest difference in these main effects.

In [ ]:
# mPC ~ pa_or_snp + covars
ys <- paste0('mPC',1:9)
xs <- c('chr17_36804493_G_T','chr14_88305056_G_A','pa_bin','pa','mod_pa','mvpa','vig_pa')
lm_runs_w_mPC_outcome <- CJ(ys, xs, covar_sets, sorted=F)

# hdl_log ~ transformed_pa_or_snp + covars
ys <- 'hdl_log'
xs <- c('chr17_36804493_G_T', 'chr14_88305056_G_A',
        'pa',       'mod_pa',       'mvpa',       'vig_pa',
        'pa_log',   'mod_pa_log',   'mvpa_log',   'vig_pa_log',
        'pa_wins',  'mod_pa_wins',  'mvpa_wins',  'vig_pa_wins',
        'pa_trunc', 'mod_pa_trunc', 'mvpa_trunc', 'vig_pa_trunc')
lm_runs_w_hdl_outcome <- CJ(ys, xs, covar_sets, sorted=F)

runs <- rbind(lm_runs_w_mPC_outcome,
              lm_runs_w_hdl_outcome)
runs <- runs[, fmlas := paste0('scale(',ys,')','~','scale(',xs,')','+',sapply(covar_sets,paste,collapse='+')) |> sapply(formula) ]

runs[
  ][, lm_coefs := Map(fmlas, f = \(fmla)
        lm(fmla,data[exam==1]) |> (\(model) summary(model)$coefficients)()
      )
  # Extract the results we want
  ][, var_coefs := mapply(lm_coefs, xs, FUN = \(coefs,x)
        coefs[grepl(x,rownames(coefs)),],
      SIMPLIFY=F)

  ][, estimate := sapply(var_coefs, '[', 'Estimate'  )
  ][, se       := sapply(var_coefs, '[', 'Std. Error')
  ][, t        := sapply(var_coefs, '[', 't value'   )
  ][, p        := sapply(var_coefs, '[', 'Pr(>|t|)'  )
  ][, l95      := estimate-(1.96*se),
  ][, u95      := estimate+(1.96*se),

  # Cleanup
  ][, covar_sets := factor(covar_sets, levels=..covar_sets, labels=names(..covar_sets))
  ][, var_coefs := NULL
  ][, lm_coefs := NULL
  ][, fmlas := NULL
]

In [ ]:
ggplot(runs[grepl('mPC',ys)], aes(y=estimate, x=covar_sets, color=covar_sets)) +
  geom_point   (position=position_dodge(width=0.2)) +
  geom_errorbar(position=position_dodge(width=0.2), aes(ymin=l95,ymax=u95), width=0) +
  geom_hline(yintercept=0, color="gray") +
  scale_x_discrete(labels=NULL, breaks=NULL) + labs(x=NULL) +
  facet_grid(rows=vars(xs), cols=vars(ys), scales="free") +
  ggtitle('Effect of various covariate sets on the estimate of `variable` in `mPC# ~ variable + covar_set`')

In [ ]:
ggplot(runs[grepl('hdl',ys)], aes(y=estimate, x=covar_sets, color=covar_sets)) +
  geom_point   (position=position_dodge(width=0.2)) +
  geom_errorbar(position=position_dodge(width=0.2), aes(ymin=l95,ymax=u95), width=0) +
  geom_hline(yintercept=0, color="gray") +
  scale_x_discrete(labels=NULL, breaks=NULL) + labs(x=NULL) +
  facet_grid(cols=vars(xs), scales="free") +
  ggtitle('Effect of various covariate sets on the estimate of `variable` in `hdl_log ~ variable + covar_set`')

In [ ]:
runs[ys=='hdl_log' &
     !grepl('chr',xs) &
     covar_sets=='+gPCs'
   ][order(-t)
   ][, xs := factor(xs,levels=xs) # So ggplot doesn't alphabetically order things
] |>
  ggplot(aes(x=xs, y=t)) +
  geom_bar(stat='identity') +
  ggtitle('T-score of different PA variables in `hdl_log ~ pa_variable + full_covar_set`')

# Covariate Distributions

In [ ]:
vars2hist_quant <- c('pa','mod_pa','mvpa','vig_pa', 'age','bmi','ahei_score','dash_score','ses_score','drinks_per_week', 'hdl','hdl_log')
data2hist_quant <- melt(data[, c('exam','sex', ..vars2hist_quant)],
  measure.vars  = vars2hist_quant,
  variable.name = 'var',
  value.name    = 'val'
)[, exam_means := mean(val,na.rm=T),            by=.(var,exam)
][, `exam_-1sd`:= exam_means - sd(val,na.rm=T), by=.(var,exam)
][, `exam_+1sd`:= exam_means + sd(val,na.rm=T), by=.(var,exam)
][, sex_means := mean(val,na.rm=T),           by=.(var,sex)
][, `sex_-1sd`:= sex_means - sd(val,na.rm=T), by=.(var,sex)
][, `sex_+1sd`:= sex_means + sd(val,na.rm=T), by=.(var,sex)
]

vars2hist_categ <- c('race','smoking','income','chr17_36804493_G_T')
data2hist_categ <- melt(data[, c('exam','sex', ..vars2hist_categ)],
  measure.vars  = vars2hist_categ,
  variable.name = 'var',
  value.name    = 'val'
)[, val := factor(val,levels=c( # So ggplot doesn't alphabetically order thigns
    '0','1','2',
    'NEVER', 'FORMER', 'CURRENT',
    'BLACK, AFRICAN-AMERICAN', 'CHINESE AMERICAN', 'HISPANIC', 'WHITE, CAUCASIAN',
    'Missing', '< 5000', '5000 - 7999', '8000 - 11999', '12000 - 15999', '16000 - 19999', '20000 - 24999', '25000 - 29999', '30000 - 34999', '35000 - 39999', '40000 - 49999', '50000 - 74999', '75000 - 99999', '100000 +', '100000 - 124999', '125000 - 149999', '150000 OR MORE'
))]

In [ ]:
ggplot(data2hist_quant, aes(x=val, fill=sex)) +
  geom_density(alpha=0.3) +
  facet_wrap(vars(var), scales = 'free') +
  geom_vline(aes(xintercept= sex_means, color=sex)) +
  geom_vline(aes(xintercept=`sex_-1sd`, color=sex), linetype='dashed') +
  geom_vline(aes(xintercept=`sex_+1sd`, color=sex), linetype='dashed')

ggplot(data2hist_quant, aes(x=val, fill=exam)) +
  geom_density(alpha=0.3) +
  facet_wrap(vars(var), scales = 'free') +
  geom_vline(aes(xintercept= exam_means, color=exam)) +
  geom_vline(aes(xintercept=`exam_-1sd`, color=exam), linetype='dashed') +
  geom_vline(aes(xintercept=`exam_+1sd`, color=exam), linetype='dashed')

In [ ]:
ggplot(data2hist_categ, aes(x=val)) + 
  geom_histogram(aes(fill=sex), stat='count', position='dodge', alpha=0.5) +
  facet_wrap(vars(var), nrow=1, scales = 'free') +
  theme_bw() + theme(axis.text.x = element_text(angle=30, hjust=0.9))